In [1]:
import os
import h5py
import pandas as pd

In [2]:
def collect_time_stamps(dir):
    """
    收集指定目录下所有以时间戳（形如HHMMSS）命名的h5文件。

    参数:
        dir (str): 目录路径。

    返回:
        List[str]: 排序后的时间戳字符串列表（如['092700', '093001']）。
    """
    time_stamps = []
    for file in os.listdir(dir):
        if file.endswith('.h5'):
            # 文件名格式为HHMMSS.h5
            time_stamp = file.split('.')[0]
            if len(time_stamp) == 6 and time_stamp.isdigit():
                time_stamps.append(time_stamp)
    time_stamps.sort()
    return time_stamps

def read_single_file(filepath, show_file_info=True):
    """
    读取单个本地因子hdf5文件，转换为DataFrame，可选是否展示文件结构与内容信息。

    参数:
        filepath (str): hdf5文件路径。
        show_file_info (bool): 是否打印文件内的所有key及部分内容，默认True。

    返回:
        pd.DataFrame: 因子数据表（列名由factorlist提供）。
    """
    hf = h5py.File(filepath, 'r')
    if show_file_info:
        print(f'{filepath} keys:')
        print(hf.keys())
        for key in hf.keys():
            print(' - ', hf[key])
            if key.startswith('codelist') or key.startswith('factorlist'):
                try:
                    content = [t.decode() for t in hf[key][:]]
                    print('   ', content)
                except:
                    content = [t[0].decode() for t in hf[key][:]]
                    print('   ', content)
    # 每个文件中只有一个时间戳的数据
    column_list = []
    try:
        column_list = [t.decode() for t in hf["factorlist"][:]]
    except:
        column_list = [t[0].decode() for t in hf["factorlist"][:]]
    df = pd.DataFrame(hf["factordata"][:], columns=column_list)
    hf.close()
    return df

def read_local_factor_data(dir, time_stamps, show_file_info=True):
    """
    读取指定目录下一组按时间戳命名的h5因子文件，结果以时间戳为key存字典。

    参数:
        dir (str): 存放h5文件的目录路径。
        time_stamps (List[str]): 需读取的文件时间戳（文件名不含后缀）。
        show_file_info (bool): 是否显示每个文件的详细信息，默认True。

    返回:
        Dict[str, pd.DataFrame]: key为时间戳，value为对应的因子数据DataFrame。
    """
    df_dict = {}
    for time_stamp in time_stamps:
        filepath = f'{dir}/{time_stamp}.h5'
        if os.path.exists(filepath):
            df = read_single_file(filepath, show_file_info)
            df_dict[time_stamp] = df
        else:
            print(f'{time_stamp} not found')
    return df_dict

In [3]:
# 读取hdf5文件
date = 20220222
dir = f'./test/factor_data/{date}/open5m_factor_demo_local'
time_stamps = collect_time_stamps(dir)
print(time_stamps)
# 读取第一个文件展示以下文件结构
read_local_factor_data(dir, time_stamps[:1])
# 读取所有文件
df_dict = read_local_factor_data(dir, time_stamps, show_file_info=False)

['092700', '093001', '093002', '093340', '093419']
./test/factor_data/20220222/open5m_factor_demo_local/092700.h5 keys:
<KeysViewHDF5 ['codelist', 'factordata', 'factorlist']>
 -  <HDF5 dataset "codelist": shape (4415,), type "|S6">
    ['000001', '000002', '000004', '000006', '000008', '000009', '000010', '000011', '000012', '000014', '000016', '000017', '000019', '000020', '000021', '000023', '000025', '000026', '000027', '000028', '000029', '000030', '000031', '000032', '000034', '000035', '000036', '000037', '000038', '000039', '000040', '000042', '000045', '000046', '000048', '000049', '000050', '000055', '000056', '000058', '000059', '000060', '000061', '000062', '000063', '000065', '000066', '000068', '000069', '000070', '000078', '000088', '000089', '000090', '000096', '000099', '000100', '000150', '000151', '000153', '000155', '000156', '000157', '000158', '000159', '000166', '000301', '000333', '000338', '000400', '000401', '000402', '000403', '000404', '000407', '000408', '0

In [4]:
print(time_stamps[0])
df_dict[time_stamps[0]]

092700


,CodeInt,last_time,deadline,demo0000_F000,demo0000_F001,demo0000_F002,demo0000_F003,demo0000_F004,demo0000_F005,demo0000_F006,...,democs00_F004,democs00_F005,demosy00_F000,demosy00_F001,demosy00_F002,demosy00_F003,demosy00_F004,demosy00_F005,demosy00_F006,demosy00_F007
0,1.0,92500000.0,93000000.0,93000000.0,92500000.0,165100.0,164299.0,800994.55,1316023.718,1.940575e+06,...,6496.0,93000000.0,163100.0,644.0,808076.0,644.0,6496.0,1.940576e+10,16.770000,16.100000
1,2.0,92500000.0,93000000.0,93000000.0,92500000.0,207000.0,204905.0,933373.88,1912526.227,9.717553e+05,...,5857.0,93000000.0,205100.0,415.0,678000.0,415.0,5857.0,9.717553e+09,20.910000,19.559999
2,4.0,92500000.0,93000000.0,93000000.0,92500000.0,224600.0,223228.0,101331.35,226200.269,1.163308e+04,...,496.0,93000000.0,220300.0,89.0,68000.0,89.0,496.0,1.163308e+08,25.860001,21.650000
3,6.0,92500000.0,93000000.0,93000000.0,92500000.0,45600.0,45092.0,111623.50,50332.904,1.349995e+05,...,487.0,93000000.0,45400.0,30.0,99800.0,30.0,487.0,1.349995e+09,4.570000,4.420000
4,8.0,92500000.0,93000000.0,93000000.0,92500000.0,28800.0,29245.0,1955441.77,571860.778,2.693783e+05,...,2382.0,93000000.0,28700.0,517.0,3559900.0,517.0,2382.0,2.693783e+09,3.100000,2.490000
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4410,603230.0,92459710.0,93000000.0,93000000.0,92459710.0,156600.0,156746.0,82675.66,129590.589,8.838100e+03,...,562.0,93000000.0,154800.0,67.0,84500.0,67.0,562.0,8.838100e+07,16.320000,15.370000
4411,688167.0,92455190.0,93000000.0,93000000.0,92455190.0,1380000.0,1364605.0,5998.16,81851.218,1.931936e+03,...,87.0,93000000.0,1358000.0,8.0,1945.0,8.0,87.0,1.931936e+07,148.869995,134.300003
4412,688210.0,92605000.0,93000000.0,93000000.0,92605000.0,365000.0,363383.0,2055.31,7468.644,1.634810e+03,...,92.0,93000000.0,0.0,0.0,0.0,0.0,92.0,1.634810e+07,36.520000,33.810001
4413,688206.0,92459020.0,93000000.0,93000000.0,92459020.0,295100.0,294943.0,8263.62,24372.967,3.628648e+03,...,165.0,93000000.0,292000.0,19.0,7800.0,19.0,165.0,3.628648e+07,31.000000,28.600000


In [5]:
# 读取hdf5文件
date = 20220222
dir = f'./test/factor_data/{date}/open5m_factor_demo_live'
time_stamps = collect_time_stamps(dir)
print(time_stamps)
# 读取第一个文件展示以下文件结构
read_local_factor_data(dir, time_stamps[:1])
# 读取所有文件
df_dict2 = read_local_factor_data(dir, time_stamps, show_file_info=False)

['092700', '093001', '093002', '093340', '093419']
./test/factor_data/20220222/open5m_factor_demo_live/092700.h5 keys:
<KeysViewHDF5 ['codelist', 'factordata', 'factorlist']>
 -  <HDF5 dataset "codelist": shape (4415,), type "|S6">
    ['000001', '000002', '000004', '000006', '000008', '000009', '000010', '000011', '000012', '000014', '000016', '000017', '000019', '000020', '000021', '000023', '000025', '000026', '000027', '000028', '000029', '000030', '000031', '000032', '000034', '000035', '000036', '000037', '000038', '000039', '000040', '000042', '000045', '000046', '000048', '000049', '000050', '000055', '000056', '000058', '000059', '000060', '000061', '000062', '000063', '000065', '000066', '000068', '000069', '000070', '000078', '000088', '000089', '000090', '000096', '000099', '000100', '000150', '000151', '000153', '000155', '000156', '000157', '000158', '000159', '000166', '000301', '000333', '000338', '000400', '000401', '000402', '000403', '000404', '000407', '000408', '00

In [6]:
# 比较live与local因子数据
# 比较两个字典中的key是否相同
name1 = 'local'
name2 = 'live'
if set(df_dict.keys()) == set(df_dict2.keys()):
    print("时间戳完全相同")
else:
    print("时间戳不完全相同")
    # 打印差异
    print(f"{name1}字典中的时间戳：", set(df_dict.keys()))
    print(f"{name2}字典中的时间戳：", set(df_dict2.keys()))
    diff1 = set(df_dict.keys()) - set(df_dict2.keys())
    diff2 = set(df_dict2.keys()) - set(df_dict.keys())
    if diff1:
        print(f"{name1} 独有的时间戳：{diff1}")
    if diff2:
        print(f"{name2} 独有的时间戳：{diff2}")
print("--------------------------------")

# 比较每个时间戳的因子数据是否相同
interact_time_stamps = set(df_dict.keys()) & set(df_dict2.keys())
for time_stamp in interact_time_stamps:
    df1 = df_dict[time_stamp].sort_values(by='CodeInt').reset_index(drop=True)
    df2 = df_dict2[time_stamp].sort_values(by='CodeInt').reset_index(drop=True)
    if df1.equals(df2):
        print(f"{time_stamp} 两个DataFrame完全相同")
    else:
        print(f"{time_stamp} 两个DataFrame不完全相同")
        # 比较df1和df2的列名是否相同
        if (set(df1.columns) == set(df2.columns)) == False:
            print("两个DataFrame的列名不完全相同")
            print(f"{name1}DataFrame的列名：", df1.columns)
            print(f"{name2}DataFrame的列名：", df2.columns)
        # 比较df1和df2的形状是否相同
        if (df1.shape == df2.shape) == False:
            print("两个DataFrame的形状不完全相同")
            print(f"{name1}DataFrame的形状：", df1.shape)
            print(f"{name2}DataFrame的形状：", df2.shape)
        # 比较每一列是否相同
        for col in df1.columns:
            if df1[col].equals(df2[col]) == False:
                print(f"{col} 两列不完全相同")
                # 找出不相等的行
                diff = (df1[col] != df2[col])
                diff_rows = diff[diff]  # 获取不相同的行
                print(f"有{len(diff_rows)}行不相同")
                print(f"不相同的行索引：", diff_rows.index.tolist())
                # 分别打印df1和df2中那一列不相同的值
                print(f"{name1}中不相同的值：")
                print(df1[col][diff].to_list())
                print(f"{name2}中不相同的值：")
                print(df2[col][diff].to_list())
    print("--------------------------------")

时间戳完全相同
--------------------------------
093002 两个DataFrame完全相同
--------------------------------
092700 两个DataFrame不完全相同
deadline 两列不完全相同
有4415行不相同
不相同的行索引： [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71, 72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 100, 101, 102, 103, 104, 105, 106, 107, 108, 109, 110, 111, 112, 113, 114, 115, 116, 117, 118, 119, 120, 121, 122, 123, 124, 125, 126, 127, 128, 129, 130, 131, 132, 133, 134, 135, 136, 137, 138, 139, 140, 141, 142, 143, 144, 145, 146, 147, 148, 149, 150, 151, 152, 153, 154, 155, 156, 157, 158, 159, 160, 161, 162, 163, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 19